In [1]:
import duckdb
import pandas as pd
import json

DB_PATH = "../data/warehouse/commerce_analytics.duckdb"

conn = duckdb.connect(DB_PATH)

conn.execute("""
SELECT table_schema, table_name
FROM information_schema.tables
ORDER BY table_schema, table_name
""").fetchdf()

conn.execute("""
DESCRIBE raw.raw_document_extractions
""").fetchdf()


sample = conn.execute("""
SELECT raw_json
FROM raw.raw_document_extractions
LIMIT 1
""").fetchone()[0]

parsed = json.loads(sample) if isinstance(sample, str) else sample

parsed.keys()


dict_keys(['extraction_metadata', 'document_identity', 'order_classification', 'transaction_identity', 'transaction_context', 'document_content_flags', 'amount_summary', 'components', 'tax_breakdown', 'entities', 'raw_detected_fields', 'data_quality', '_pipeline_metadata'])

In [5]:
sample = conn.execute("""
SELECT *
FROM raw.raw_document_extractions
LIMIT 1
""").fetchone()

import json
from pprint import pprint

pprint(sample)

('fa7ab6177898f339',
 'e3a9b76b-76b3-4ed8-8a13-251777d52b0b',
 'fa7ab6177898f339c0ddd08888891946ba781885f7fcf5923e0db7a3ef3984d1',
 '585529e0ca7c2296bcfc5bd7e5ae975cecf464111cf831c753883e77935b23d7',
 'invoice_001.pdf',
 'data/invoices/invoice_001.pdf',
 'invoice_001_extraction.json',
 '/Users/amansingh/Desktop/ai_commerce_analytics/data/raw_json/ai_extractions/invoice_001_extraction.json',
 datetime.datetime(2026, 5, 31, 19, 37, 28, 624776),
 '{"extraction_metadata": {"source_platform": "Wolt", "document_type": '
 '"platform_fee_invoice", "document_type_confidence": 0.95, '
 '"document_pattern": "split_platform_fee_document", "document_language": '
 '"en", "country_or_market": "DEU", "currency": "EUR", "source_file_name": '
 'null, "document_hash": null, "source_page_numbers": [1], "extraction_notes": '
 '["Document contains platform fees, tip, service fee, and discount but no '
 'product items.", "Seller details correspond to platform legal entity."], '
 '"extraction_model": {"provid

In [3]:
import duckdb

conn = duckdb.connect("../data/warehouse/commerce_analytics.duckdb")

print(
    conn.execute("""
    select table_schema,
           table_name
    from information_schema.tables
    order by table_schema, table_name
    """).fetchdf()
)

conn.close()

  table_schema                table_name
0          raw  raw_document_extractions
1          raw         raw_file_registry
2          raw         raw_ingestion_log
3      staging             stg_documents


In [2]:
import duckdb

conn = duckdb.connect("../data/warehouse/commerce_analytics.duckdb")

conn.execute("drop view if exists main.stg_documents")
conn.execute("drop view if exists main_staging.stg_documents")

conn.close()

print("cleanup complete")

cleanup complete


In [1]:
import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)

conn = duckdb.connect("../data/warehouse/commerce_analytics.duckdb")

df = conn.execute("""
select
    residence_city,
    contains_alcohol,
    count(*) as order_count
from marts.fct_orders
group by 1, 2
order by 1, 2
""").fetchdf()

conn.close()

df

,residence_city,contains_alcohol,order_count
0,Berlin,False,2
1,Berlin,True,3


In [8]:
conn = duckdb.connect("../data/warehouse/commerce_analytics.duckdb")

df = conn.execute("""
select distinct residence_city,order_category
from marts.fct_orders
order by 1
""").fetchdf()

conn.close()

df

,residence_city,order_category
0,Berlin,convenience
1,Berlin,restaurant
2,Berlin,unknown


In [10]:
conn.close()